# TabPFN v2 — Telco Churn

TabPFN v2 é um modelo pré-treinado de in-context learning para tabular data.
Suporta até ~10k amostras de treino sem subsample. Apenas `fit` + `predict`.


In [1]:
import sys, os
sys.path.insert(0, '.')
import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from model_utils import load_data, compute_metrics, save_results, print_scores

## 0. Instalar TabPFN

In [2]:
from tabpfn import TabPFNClassifier
import tabpfn
print(f'TabPFN version: {tabpfn.__version__}')


TabPFN version: 2.2.1


## 1. Carregar dados

In [3]:
X_train, y_train, X_val, y_val, X_test, y_test = load_data()
print(f'Train: {X_train.shape}  Val: {X_val.shape}  Test: {X_test.shape}')

Xtr_np  = X_train.values.astype('float32');  ytr_np  = y_train.values.astype(int)
Xvl_np  = X_val.values.astype('float32');    yvl_np  = y_val.values.astype(int)
Xte_np  = X_test.values.astype('float32');   yte_np  = y_test.values.astype(int)


Train: (7242, 23)  Val: (1057, 23)  Test: (1057, 23)


## 2. Fit + Predict (dataset completo)


In [4]:
print(f'Train: {len(Xtr_np)} amostras (dataset completo, sem subsample)')
print(f'Churn rate treino: {ytr_np.mean():.3f}')


Train: 7242 amostras (dataset completo, sem subsample)
Churn rate treino: 0.500


## 3. Fit + Predict


In [5]:
clf = TabPFNClassifier(device='cuda', n_estimators=32)
clf.fit(Xtr_np, ytr_np)
print('TabPFN v2 fit concluído')

probs = clf.predict_proba(Xte_np)[:, 1]
preds = (probs >= 0.5).astype(int)
print(f'Predictions geradas: {len(preds)} amostras de teste')


TabPFN v2 fit concluído


Predictions geradas: 1057 amostras de teste


## 4. Avaliar no teste + salvar artefatos

In [6]:
scores = compute_metrics(yte_np, preds, probs)
print('Scores no teste:')
print_scores(scores)

os.makedirs('results/tabpfn', exist_ok=True)
joblib.dump(clf, 'results/tabpfn/model.pkl')

save_results('tabpfn', yte_np, preds, probs, scores)
print(f'\nNota: TabPFN v2, treino completo com {len(Xtr_np)} amostras.')


Scores no teste:
  accuracy               0.7928
  balanced_accuracy      0.7072
  precision              0.6309
  recall                 0.5250
  f1                     0.5731
  auroc                  0.8363
  ks                     0.5403


Saved → results/tabpfn

Nota: TabPFN v2, treino completo com 7242 amostras.
